# Neuron sensitivity in weight space

This notebook follows the local experiment in small steps. It uses fixed seeds, reloads the baseline before each intervention, and records measurements instead of assuming weight magnitude determines importance.

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from neuron_sensitivity.data import load_mnist
from neuron_sensitivity.model import choose_device, seed_everything
from neuron_sensitivity.train import train_model
from neuron_sensitivity.experiment import run_experiments
from neuron_sensitivity.plots import make_plots

DEVICE = choose_device('auto')
OUT = Path('results')
print('device:', DEVICE)

## Load data and train

The first call downloads standard MNIST files to `../data` if needed. Training metrics and a baseline checkpoint are saved before any intervention.

In [ ]:
train_data, test_data = load_mnist('data')
model, baseline_accuracy, training_metrics = train_model(
    train_data, test_data, OUT, DEVICE, seed=42, epochs=5
)
print(f'Baseline test accuracy: {baseline_accuracy:.3%}')
display(training_metrics)

## Inspect the intervention definitions

Noise adds a vector whose L2 norm is a selected fraction of the original row norm. Scaling multiplies the row by `1 + strength`. Ablation zeros both incoming weights and bias so the ReLU unit contributes zero. Bias stays fixed for noise and scaling.

In [ ]:
from neuron_sensitivity.perturb import perturb_neuron

for layer in ('fc1', 'fc2'):
    print(layer, 'neurons:', getattr(model, layer).out_features)

## Run the full test set

Each row measures one neuron intervention on the same unmodified MNIST test set. Gaussian noise has three fixed-seed repeats at each strength. This is the computationally expensive part; `sample_limit=1000` can be used for a smoke run, but final reporting should use all 10,000 test images.

In [ ]:
results = run_experiments(model, test_data, OUT, DEVICE,
                           strengths=(0.1, 0.5, 1.0), noise_repeats=3)
make_plots(results, OUT / 'plots')
display(results.head())
print('measurements:', len(results))

## Summarize and inspect

Compare the layers and methods from the measured CSV. Interpret correlations as associations, and report negative or mixed findings as well as strong effects.

In [ ]:
summary = (results.groupby(['layer', 'method', 'strength'], as_index=False)
           .agg(mean_accuracy=('perturbed_accuracy', 'mean'),
                mean_flip_pct=('prediction_flip_pct', 'mean'),
                mean_probability_change=('mean_abs_predicted_class_probability_change', 'mean')))
display(summary)
display(results[results.method == 'ablation'].groupby('layer').agg(
    mean_flips=('prediction_flip_pct', 'mean'),
    median_flips=('prediction_flip_pct', 'median'),
    mean_weight_norm=('neuron_weight_norm', 'mean')))